# 04 - Big Data Processing με Apache Spark

Το dataset μας (3M+ γραμμές) αποτελεί ένα πραγματικό **Big Data** πρόβλημα. Σε production scale (δεκάδες εκατομμύρια εγγραφές), το Pandas δεν αρκεί — χρειαζόμαστε **Apache Spark**.

Αυτό το notebook δείχνει πώς θα εκτελούσαμε το preprocessing μας **κατανεμημένα** με PySpark.


## 1. Εγκατάσταση PySpark


In [ ]:
import sys
!{sys.executable} -m pip install pyspark -q
print('PySpark installed')


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

spark = (
    SparkSession.builder
    .appName('SalesForecasting')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')

print(f'Spark version: {spark.version}')
print('SparkSession OK')


## 2. Φόρτωση Δεδομένων ως Spark DataFrames


In [ ]:
train_sdf = (
    spark.read
    .csv('../data/raw/train.csv', header=True, inferSchema=True)
    .withColumn('date', F.to_date('date'))
)

stores_sdf = spark.read.csv('../data/raw/stores.csv', header=True, inferSchema=True)

oil_sdf = (
    spark.read
    .csv('../data/raw/oil.csv', header=True, inferSchema=True)
    .withColumn('date', F.to_date('date'))
    .withColumnRenamed('dcoilwtico', 'oil_price')
)

print('=== train ===')
train_sdf.printSchema()
print(f'Rows: {train_sdf.count():,}')
print(f'Partitions: {train_sdf.rdd.getNumPartitions()}')
train_sdf.show(5)


## 3. Spark SQL — Exploratory Queries

Αθροίσεις και αναλύσεις **σε scale** — ο Spark εκτελεί αυτά τα queries κατανεμημένα.


In [ ]:
train_sdf.createOrReplaceTempView('train')
stores_sdf.createOrReplaceTempView('stores')

print('=== Top 10 Families by Total Sales ===')
spark.sql("""
    SELECT family,
           ROUND(SUM(sales), 0) AS total_sales,
           COUNT(*)             AS num_records,
           ROUND(AVG(sales), 3) AS avg_sales
    FROM   train
    GROUP  BY family
    ORDER  BY total_sales DESC
    LIMIT  10
""").show()


In [ ]:
print('=== Monthly Sales Trend (2013-2017) ===')
spark.sql("""
    SELECT YEAR(date)  AS year,
           MONTH(date) AS month,
           ROUND(SUM(sales), 0)  AS total_sales,
           ROUND(AVG(sales), 4)  AS avg_sales
    FROM   train
    GROUP  BY YEAR(date), MONTH(date)
    ORDER  BY year, month
""").show(30)


In [ ]:
print('=== Sales by Store Type (JOIN) ===')
spark.sql("""
    SELECT s.type,
           ROUND(SUM(t.sales), 0)  AS total_sales,
           ROUND(AVG(t.sales), 4)  AS avg_sales,
           COUNT(*)                 AS records
    FROM   train t
    JOIN   stores s ON t.store_nbr = s.store_nbr
    GROUP  BY s.type
    ORDER  BY total_sales DESC
""").show()

print('=== Promotion Impact ===')
spark.sql("""
    SELECT CASE WHEN onpromotion > 0 THEN 'On Promotion' ELSE 'Normal' END AS promo,
           ROUND(AVG(sales), 4) AS avg_sales,
           COUNT(*)             AS count
    FROM   train
    GROUP  BY CASE WHEN onpromotion > 0 THEN 'On Promotion' ELSE 'Normal' END
""").show()


## 4. Spark Preprocessing Pipeline

Αντίστοιχο με το notebook 02, αλλά **scalable** σε οποιοδήποτε cluster.


In [ ]:
# === Oil price forward-fill με Window Function ===
print('Oil price fill με Window...')

date_range_sdf = spark.sql("""
    SELECT explode(sequence(
        DATE('2013-01-01'),
        DATE('2017-08-15'),
        INTERVAL 1 DAY
    )) AS date
""")

oil_full_sdf = date_range_sdf.join(oil_sdf, on='date', how='left')

w_fill = Window.orderBy('date').rowsBetween(Window.unboundedPreceding, 0)
oil_filled_sdf = oil_full_sdf.withColumn(
    'oil_price',
    F.last('oil_price', ignorenulls=True).over(w_fill)
)

print(f'Oil rows (full calendar): {oil_filled_sdf.count()}')
print(f'NaN after fill: {oil_filled_sdf.filter(F.col("oil_price").isNull()).count()}')
oil_filled_sdf.show(5)


In [ ]:
# === Join: train + stores + oil ===
print('Joining train + stores + oil...')

df_sdf = (
    train_sdf
    .join(stores_sdf, on='store_nbr', how='left')
    .join(oil_filled_sdf, on='date', how='left')
)

# === Date features ===
df_sdf = (
    df_sdf
    .withColumn('year',        F.year('date'))
    .withColumn('month',       F.month('date'))
    .withColumn('day',         F.dayofmonth('date'))
    .withColumn('day_of_week', F.dayofweek('date'))
    .withColumn('is_weekend',  (F.dayofweek('date').isin([1, 7])).cast('int'))
    .withColumn('quarter',     F.quarter('date'))
)

# === Sales classification (same thresholds as preprocessing notebook) ===
df_sdf = df_sdf.withColumn(
    'sales_class',
    F.when(F.col('sales') <= 4.0,   'Low')
     .when(F.col('sales') <= 150.0, 'Medium')
     .otherwise('High')
)

print(f'Final Spark DataFrame columns ({len(df_sdf.columns)}): {df_sdf.columns}')
df_sdf.show(3)


## 5. Aggregations & Business Queries


In [ ]:
# Store-monthly aggregation
print('=== Store-Monthly Aggregation ===')
store_monthly = (
    df_sdf
    .groupBy('store_nbr', 'year', 'month', 'type', 'state')
    .agg(
        F.round(F.sum('sales'), 2).alias('total_sales'),
        F.round(F.avg('sales'), 4).alias('avg_sales'),
        F.count('*').alias('records'),
        F.sum('onpromotion').alias('promotions')
    )
    .orderBy('store_nbr', 'year', 'month')
)
print(f'Rows: {store_monthly.count():,}')
store_monthly.show(10)


In [ ]:
# Pandas visualization from Spark query
monthly_pdf = spark.sql("""
    SELECT YEAR(date)  AS year,
           MONTH(date) AS month,
           SUM(sales)  AS total_sales
    FROM   train
    GROUP  BY YEAR(date), MONTH(date)
    ORDER  BY year, month
""").toPandas()

monthly_pdf['period'] = pd.to_datetime(
    monthly_pdf[['year', 'month']].assign(day=1)
)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly_pdf['period'], monthly_pdf['total_sales'] / 1e6,
        marker='o', markersize=3, linewidth=1.5, color='steelblue')
ax.fill_between(monthly_pdf['period'], monthly_pdf['total_sales'] / 1e6, alpha=0.12)
ax.set_title('Μηνιαίες Πωλήσεις (από Spark SQL Query)', fontsize=13)
ax.set_xlabel('Μήνας')
ax.set_ylabel('Total Sales (εκατ.)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Partitioning & Performance

Το Spark χωρίζει τα δεδομένα σε **partitions** και τα επεξεργάζεται παράλληλα σε πολλαπλούς κόμβους.


In [ ]:
print(f'Default partitions: {df_sdf.rdd.getNumPartitions()}')

# Repartition by year — optimal για time-based queries
df_repartitioned = df_sdf.repartition(4, 'year')
print(f'After repartition by year: {df_repartitioned.rdd.getNumPartitions()}')

# Cache frequently used DataFrame
df_sdf.cache()
print('DataFrame cached in memory ✓')

# Show execution plan
print('\nExecution Plan (aggregation):')
(
    df_sdf.select('store_nbr', 'sales', 'year')
    .groupBy('year')
    .agg(F.sum('sales'))
    .explain(False)
)


## 7. Αποθήκευση Processed Data σε Parquet


In [ ]:
import os
os.makedirs('../data/spark_output', exist_ok=True)

# Αποθήκευση partitioned by year
try:
    df_sdf.write.mode('overwrite').partitionBy('year').parquet('../data/spark_output/processed')
    print('Spark output saved to data/spark_output/processed/ ✓')
except Exception as e:
    print(f'Save failed: {e}')
    print('(Normal in local mode with limited memory)')


## 8. Pandas vs Spark — Σύγκριση

| Χαρακτηριστικό | Pandas | Apache Spark |
|---|---|---|
| Μέγεθος δεδομένων | GBs (RAM) | PBs (distributed) |
| Ταχύτητα (μικρά data) | Πολύ γρήγορο | Overhead (~30s startup) |
| Ταχύτητα (μεγάλα data) | Αδύνατο | Γρήγορο (horizontal scaling) |
| APIs | Pythonic, interactive | DataFrame + SQL |
| Deployment | Laptop | Cluster (Hadoop/Kubernetes/Cloud) |
| Machine Learning | scikit-learn | MLlib, Spark NLP |
| Κόστος | Δωρεάν | Cloud: $$/hr per node |

**Για το project μας:** Τα 3M rows χωράνε άνετα σε Pandas (286 MB). Αν το dataset ήταν **100M+ rows** (π.χ. εβδομαδιαία granularity ή πολλά stores), το Spark θα ήταν απαραίτητο.


In [ ]:
spark.stop()
print('SparkSession stopped ✓')


## 9. Συμπεράσματα

**Τι δείξαμε:**
1. Φόρτωση και schema inference CSV αρχείων με Spark
2. SparkSQL queries για aggregations και JOINs σε 3M+ γραμμές
3. Window functions για forward-fill time series data
4. Equivalent preprocessing pipeline (joins, date features, classification)
5. Partitioning για optimal performance
6. Αποθήκευση σε partitioned Parquet format

**Real-world scenario:** Σε production με δεκάδες εκατομμύρια εγγραφές, το Spark εκτελείται σε cluster (AWS EMR, Google Dataproc, Azure HDInsight) — το ίδιο PySpark code τρέχει χωρίς αλλαγές.
